# 🚀 OPERA CHAIR Benchmark on Kaggle (2× NVIDIA T4 GPUs)

This notebook runs the **CHAIR (Caption Hallucination Assessment with Image Relevance)** benchmark on **LLaVA-1.5-7B** or **Qwen2-VL-7B-Instruct** using the **OPERA** hallucination mitigation method.

### ⏱️ Latency & Efficiency Profiling (10 Samples):
- Configured to evaluate **10 images sampled from COCO val2014 (Seed 2027)** to measure and calculate the **average runtime per sample** during detailed image captioning.
- Evaluates both hallucination mitigation performance (`CHAIRs`, `CHAIRi`, `Recall`, `Caption Length`) and inference speed (`avg_time_per_sample_s`, throughput `samples/sec`).
- Uses `torch.cuda.synchronize()` for microsecond-level GPU execution timing.

### Key Technical Requirements & Configurations:
- **Hardware:** 2× NVIDIA T4 GPUs (32GB VRAM total) with `device_map="auto"`.
- **Precision:** BF16 Full Precision (`torch.bfloat16`) with automatic FP16 fallback. **No quantization** (no 4-bit / 8-bit bitsandbytes).
- **Prompt:** `"Describe this image."`
- **Decoding Strategy:** Strictly **Greedy Decoding** (`do_sample=False`, `temperature=0.0`, `max_new_tokens=128`).
- **Supported Models:** Both **LLaVA-1.5-7B** (`MODEL = "llava"`) and **Qwen2-VL-7B-Instruct** (`MODEL = "qwen2vl"`).
- **Standalone Evaluation:** Uses pre-tokenized `chair.pkl` evaluator for instantaneous metric computation without downloading external annotation files.

In [ ]:
# Cell 1: Environment Setup & Dependencies Installation
# 1. Gỡ bỏ torchaudio để giải quyết dứt điểm xung đột CUDA version mismatch
!pip uninstall -y -q torchaudio

# 2. Cài đặt các thư viện cần thiết (không cài torchvision/torch để tránh lệch CUDA Kaggle)
!pip install -q --no-cache-dir \
    "transformers>=4.45.0" \
    "accelerate>=0.26.0" \
    sentencepiece \
    protobuf \
    tiktoken \
    qwen_vl_utils \
    pyyaml \
    tqdm \
    nltk \
    huggingface_hub \
    pandas

print("✅ Dependencies successfully installed!")
from transformers import AutoProcessor
print("✅ AutoProcessor import verified successfully!")


### 🔐 Step 2: HuggingFace Authentication
Access HuggingFace Hub to download pretrained models (`llava-hf/llava-1.5-7b-hf` or `Qwen/Qwen2-VL-7B-Instruct`) using Kaggle Secrets (`HF_TOKEN`).

In [ ]:
# Cell 2: HuggingFace Login via Kaggle Secrets
import os
from huggingface_hub import login

try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("HF_TOKEN")
    login(token=hf_token)
    print("✅ HuggingFace login successful!")
except Exception as e:
    print(f"⚠️ Note on Kaggle Secret retrieval: {e}")
    print("If running outside Kaggle or without Kaggle Secrets, login manually via: huggingface-cli login")


### 📂 Step 3: Clone OPERA Repository
Clone the repository to `/kaggle/working` and change working directory to `opera_experiments`.

In [ ]:
# Cell 3: Clone OPERA repository and switch to experiments directory
import os
import sys

os.chdir("/kaggle/working")

# If repo not already present, clone it; if present, pull latest code
if not os.path.exists("OPERA"):
    !git clone https://github.com/ntmy12/OPERA.git
    print("✅ OPERA repository cloned successfully!")
else:
    print("ℹ️ OPERA repository already exists. Pulling latest code...")
    !git -C OPERA pull

%cd /kaggle/working/OPERA/opera_experiments
print(f"Current Working Directory: {os.getcwd()}")


### 🖥️ Step 4: Hardware & COCO Dataset Verification
Verify that 2× NVIDIA T4 GPUs are active and that the COCO val2014 dataset is properly detected.

In [ ]:
# Cell 4: Hardware Verification & Dataset Auto-Detection
import os
import glob
import torch
import yaml

print("=" * 70)
print("         HARDWARE & ENVIRONMENT VERIFICATION")
print("=" * 70)

# 1. GPU Check
gpu_count = torch.cuda.device_count()
print(f"🖥️ CUDA Available: {torch.cuda.is_available()}")
print(f"🖥️ Total GPUs Detected: {gpu_count}")

assert gpu_count >= 1, "❌ Error: No GPU detected. Please enable GPU accelerator in Kaggle Notebook Settings!"

for i in range(gpu_count):
    name = torch.cuda.get_device_name(i)
    total_mem = torch.cuda.get_device_properties(i).total_memory / (1024**3)
    print(f"   - GPU {i}: {name} ({total_mem:.2f} GB VRAM)")

if gpu_count >= 2:
    print("✅ 2× T4 GPUs available! device_map='auto' will shard across both cards.")
else:
    print(f"ℹ️ Running on {gpu_count} GPU.")

# Check BF16 capability
bf16_ok = torch.cuda.is_bf16_supported()
print(f"⚡ Native BF16 Supported: {bf16_ok}")
if not bf16_ok:
    print("ℹ️ T4 GPU architecture uses FP16 native cores; wrapper will auto-resolve optimal precision.")

# 2. COCO val2014 Dataset Validation
coco_candidates = [
    "/kaggle/input/datasets/biminhco/val2014/val2014",
    "/kaggle/input/datasets/biminhco/val2014",
    "/kaggle/input/val2014/val2014",
    "/kaggle/input/coco-2014-val/val2014",
    "/kaggle/input/coco-val2014/val2014",
    "/kaggle/input/coco2014/val2014",
]

coco_dir = None
for p in coco_candidates:
    if os.path.isdir(p):
        coco_dir = p
        break

if coco_dir is None:
    print("🔍 Scanning /kaggle/input/ for COCO_val2014_*.jpg...")
    for root, _, files in os.walk("/kaggle/input/"):
        for f in files[:50]:
            if f.startswith("COCO_val2014_") and f.lower().endswith((".jpg", ".jpeg", ".png")):
                coco_dir = root
                break
        if coco_dir:
            break

assert coco_dir is not None, "❌ Error: COCO val2014 image folder not found in /kaggle/input/! Please attach dataset."
image_count = len(glob.glob(os.path.join(coco_dir, "COCO_val2014_*.jpg")))
print(f"✅ COCO val2014 Image Directory: {coco_dir} ({image_count} images found)")

# 3. CHAIR Manifest & Evaluator Cache Verification
manifest_file = "benchmarks/chair/selected_chair_val2014_seed2027.json"
cache_file = "benchmarks/chair/chair.pkl"
print(f"✅ CHAIR Manifest exists: {os.path.isfile(manifest_file)}")
print(f"✅ CHAIR Evaluator Cache exists: {os.path.isfile(cache_file)}")
print("=" * 70)


### ⚡ Step 5: Execute CHAIR Benchmark on 10 Samples
Run benchmark on **10 images** sampled from COCO val2014 (Seed 2027) to calculate the **average runtime per sample**.
- Model is loaded **once** onto GPU(s).
- Supports both models: set `MODEL = "llava"` (LLaVA-1.5-7B) or `MODEL = "qwen2vl"` (Qwen2-VL-7B-Instruct).
- Uses `--num_samples 10` and `max_new_tokens 128` (Greedy Decoding) for accurate latency profiling.

In [ ]:
# Cell 5: Run CHAIR Benchmark with OPERA (10 samples for latency profiling)
# Choose model:
#   - 'llava'   : LLaVA-1.5-7B (llava-hf/llava-1.5-7b-hf)
#   - 'qwen2vl' : Qwen2-VL-7B-Instruct (Qwen/Qwen2-VL-7B-Instruct)
MODEL = "llava"  # 👈 Change to 'qwen2vl' if evaluating Qwen2-VL

# Number of samples to evaluate for timing test
NUM_SAMPLES = 10  # 👈 Evaluates 10 images to calculate average runtime per sample

print(f"🚀 Starting CHAIR Benchmark with OPERA")
print(f"   Model       : {MODEL.upper()}")
print(f"   Samples     : {NUM_SAMPLES} images (Seed 2027)")
print(f"   Prompt      : 'Describe this image.'")
print(f"   Max Tokens  : 128 tokens, Greedy Decoding")

# Execute CHAIR benchmark for 10 samples
!python benchmarks/chair/run_chair.py \
    --model {MODEL} \
    --use_opera \
    --num_samples {NUM_SAMPLES} \
    --max_new_tokens 128 \
    --prompt "Describe this image." \
    --dtype bf16 \
    --device auto \
    --seed 2027 \
    --coco_dir "{coco_dir}"


### 📊 Step 6: Results Consolidation & Average Runtime Calculation
Aggregate the metrics from the CHAIR benchmark run (`CHAIRs`, `CHAIRi`, `Recall`, `Caption Length`) and display the **Average Runtime Per Sample**.

In [ ]:
# Cell 6: Summarize, Calculate Average Runtime, and Display Results
import os
import glob
import json
import pandas as pd

# Find latest results folder for the selected model
result_dirs = sorted(glob.glob(f"results/{MODEL}_chair_opera_*"))

if not result_dirs:
    print("❌ No results found. Please check Cell 5 output for errors.")
else:
    latest_run = result_dirs[-1]
    print(f"📁 Results Directory: {latest_run}\n")

    metrics_file = os.path.join(latest_run, "metrics.json")
    if os.path.exists(metrics_file):
        with open(metrics_file, 'r', encoding='utf-8') as f:
            metrics = json.load(f)

        row = [{
            "Model": MODEL.upper(),
            "Method": "OPERA",
            "Samples": metrics.get("num_evaluated", metrics.get("total_evaluated", 0)),
            "CHAIRs (%)": metrics.get("CHAIRs", 0.0),
            "CHAIRi (%)": metrics.get("CHAIRi", 0.0),
            "Recall (%)": metrics.get("Recall", 0.0),
            "Caption Length": metrics.get("Caption_Length", 0.0),
            "Avg Time/Sample (s)": metrics.get("avg_time_per_sample_s", "N/A"),
            "Total Time (s)": metrics.get("total_inference_time_s", "N/A"),
        }]

        df = pd.DataFrame(row)
        print("=" * 95)
        print(f"       CHAIR BENCHMARK RESULTS (10 SAMPLES) | Model: {MODEL.upper()} | Method: OPERA")
        print("=" * 95)
        display(df)
        print("=" * 95)

        # Detailed Timing Summary
        tot_samples = metrics.get("num_evaluated", metrics.get("total_evaluated", 0))
        tot_time = metrics.get("total_inference_time_s", 0.0)
        avg_time = metrics.get("avg_time_per_sample_s", 0.0)
        print("\n" + "⏱️ " + "=" * 65)
        print(f"   AVERAGE RUNTIME PER SAMPLE SUMMARY | {MODEL.upper()} + OPERA (CHAIR)")
        print("=" * 68)
        print(f"   • Total Images Evaluated  : {tot_samples} samples (Seed 2027)")
        print(f"   • Total Inference Time    : {tot_time:.2f} seconds")
        print(f"   • AVERAGE TIME / SAMPLE   : {avg_time:.4f} seconds / sample")
        if avg_time > 0:
            print(f"   • Throughput              : {1.0 / avg_time:.2f} samples / second")
        print("=" * 68)
    else:
        print(f"⚠️ metrics.json not found in {latest_run}")

    # Optional: Display cross-model comparison if both models were run
    all_summary_files = sorted(glob.glob("results/*_chair_opera_*/summary_metrics.json"))
    if len(all_summary_files) > 1:
        comp_rows = []
        for s_file in all_summary_files:
            folder = os.path.basename(os.path.dirname(s_file))
            m_tag = folder.split("_chair_")[0]
            try:
                with open(s_file, 'r', encoding='utf-8') as f:
                    s_data = json.load(f)
                m_info = s_data.get("metrics", {})
                comp_rows.append({
                    "Model": m_tag.upper(),
                    "Samples": m_info.get("num_evaluated", m_info.get("total_evaluated", 0)),
                    "CHAIRs (%)": m_info.get("CHAIRs", 0.0),
                    "CHAIRi (%)": m_info.get("CHAIRi", 0.0),
                    "Recall (%)": m_info.get("Recall", 0.0),
                    "Avg Time/Sample (s)": m_info.get("avg_time_per_sample_s", 0.0),
                    "Total Time (s)": m_info.get("total_inference_time_s", 0.0),
                    "Results Dir": folder
                })
            except Exception:
                pass
        if comp_rows:
            print("\n📊 Cross-Model Runtime Comparison (CHAIR):")
            display(pd.DataFrame(comp_rows))
